In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
import os
os.chdir("/content/drive/My Drive/YOLOV11HyperparameterSearch/")

In [3]:
!pip install -U ultralytics
!pip install -U ultralytics "ray[tune]"
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 MB 37.5 MB/s eta 0:00:00


In [4]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: czc0710 (czc0710-universidad-san-francisco-de-quito) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# %%
from yolo_config import Config
from ultralytics import YOLO
from ray import tune
import torch
import ray
# %%

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
if __name__ == '__main__':
    Config.set_gpu_settings()

    ray.shutdown()
    ray.init(num_gpus=1)

    # Load a YOLO11n model
    MODEL_PATH = "yolo11n.pt"
    model = YOLO(MODEL_PATH)

    os.chdir("/content/drive/My Drive/")
    YAML_PATH = os.path.join("/content/drive/My Drive/", Config.DATASET, "data.yaml")

    # Space for hyperparameters search
    search_space = {
        'lr0': tune.uniform(5e-5, 5e-3),         # Tasa de aprendizaje inicial
        'lrf': tune.uniform(0.05, 0.5),          # Factor de tasa de aprendizaje final
        'weight_decay': tune.uniform(5e-5, 5e-4), # Regularización L2
        'box': tune.uniform(0.02, 0.12),         # Peso de pérdida de caja
        'cls': tune.uniform(0.3, 1.5),           # Peso de pérdida de clase
        'scale': tune.uniform(0.1, 0.4),         # Aumento de escala
        'translate': tune.uniform(0.0, 0.15),    # Aumento de traslación
        'mosaic': tune.uniform(0.1, 0.5)         # Probabilidad de mosaico
    }

    # Start tuning hyperparameters for YOLO11n training on the CBIS-DDSM dataset
    result_grid = model.tune(
        data=YAML_PATH,
        device=Config.DEVICE,
        epochs=50,
        space=search_space,
        use_ray=True)

Output hidden; open in https://colab.research.google.com to view.